# Missing Information: Do Images Help When Measurements Are Unavailable?

## Objective

Notebook 07 found no significant gain from frozen image embeddings when all 24 features were available. This notebook tests a narrower hypothesis:

> Image embeddings improve grading when slit-lamp findings or Konan measurements are unavailable.

## Pre-specified Comparisons

Every comparison is Hybrid (tabular + ResNet-50 → PCA 16) minus Tabular, using the same RF hyperparameters, donor-grouped folds and evaluation tissues as notebook 07.

| ID | Type | Condition | Evaluation tissues |
|---|---|---|---|
| C1 | Natural | Slit-lamp findings missing in the records | Tissues with no slit-lamp findings |
| C2 | Natural | Konan measurements missing in the records | Tissues with no Konan measurements |
| C3 | Masking (test only) | Primary models; slit-lamp findings removed from held-out tissues | Tissues that originally had findings |
| C4 | Masking (test only) | Primary models; Konan measurements removed from held-out tissues | Tissues that originally had measurements |
| C5 | Masking (train + test) | Slit-lamp findings never used (pre-examination scenario) | All tissues |

- **Primary metric:** macro F1. Balanced accuracy and per-grade recall (especially grade B) are also reported.
- **Uncertainty:** corrected repeated-CV t-test on fold-level paired differences (as in notebook 07). Holm correction is applied across the five macro F1 comparisons.
- **Natural vs artificial missingness are reported separately.** Natural subgroups may differ in other ways (e.g. time period); masking holds the tissues fixed but creates an artificial pattern.
- Masked values are set to missing, so they pass through the same imputation as naturally missing values: −1 / `__MISSING__` for slit-lamp features, training median for Konan features.
- All preprocessing, PCA and RF fitting happen inside each donor-grouped training fold. Nothing is tuned here.

In [1]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

sys.path.insert(0, str(Path("../../scripts").resolve()))
import modeling_setup as ms

SEED = ms.RANDOM_STATE
IMAGE_DATA_DIR = ms.PROCESSED_DIR.parent.parent / "image" / "data" / "processed"
EMBEDDING_PATH = IMAGE_DATA_DIR / "embeddings" / "resnet50_imagenet_fullres.npz"
NB07_OOF_PATH = ms.PROCESSED_DIR / "hybrid_oof_predictions.csv"
PCA_COMPONENTS = 16

# Identical to notebook 07.
RF_PARAMS = dict(
    n_estimators=300, max_depth=15, min_samples_split=5, min_samples_leaf=1,
    max_features="sqrt", class_weight="balanced", random_state=SEED, n_jobs=-1,
)

SLIT_LAMP = ms.CLINICAL_FEATURES
KONAN = ms.NUMERIC_FEATURES

## 1. Cohort, Folds and Embeddings

In [2]:
def normalize_id(value):
    return " ".join(str(value).split()).replace("NEB- ", "NEB-").casefold()

labelled = ms.load_labelled_data()
folds = ms.get_donor_folds(labelled)

manifest = pd.read_csv(IMAGE_DATA_DIR / "clean_image_manifest.csv")
review_ids = {normalize_id(i) for i in ["NEB-2024-08-16259 R", "NEB-2025-09-17471 R"]}
manifest["Match_ID"] = manifest["Tissue_ID"].map(normalize_id)
manifest = manifest[~manifest["Match_ID"].isin(review_ids)]

labelled["Match_ID"] = labelled["Tissue_ID"].map(normalize_id)
cohort = (
    labelled
    .merge(manifest[["Match_ID", "Selected_Image"]], on="Match_ID",
           how="inner", validate="one_to_one")
    .merge(folds, on="Tissue_ID", how="left", validate="one_to_one")
    .reset_index(drop=True)
)

if not EMBEDDING_PATH.exists():
    raise FileNotFoundError("Run 07_hybrid_evaluation.ipynb first to create the embeddings.")
cached = np.load(EMBEDDING_PATH, allow_pickle=True)
assert list(cached["tissue_ids"]) == list(cohort["Tissue_ID"]), "Embedding order mismatch"
embeddings = cached["embeddings"]

EMBEDDING_COLUMNS = [f"emb_{i}" for i in range(embeddings.shape[1])]
X = pd.concat([
    cohort[ms.ALL_FEATURES].reset_index(drop=True),
    pd.DataFrame(embeddings, columns=EMBEDDING_COLUMNS),
], axis=1)
y = cohort[ms.TARGET].to_numpy()

print("Cohort:", len(cohort), "tissues,", cohort["Donor_Group"].nunique(), "donors")
print("Embeddings:", embeddings.shape)

Cohort: 1009 tissues, 770 donors
Embeddings: (1009, 2048)


## 2. Missingness Groups

A tissue counts as **missing slit-lamp findings** when all 18 slit-lamp features are missing, and as **missing Konan measurements** when all six morphology features are missing. Partial missingness is reported for completeness.

In [3]:
n_slit_present = cohort[SLIT_LAMP].notna().sum(axis=1)
n_konan_present = cohort[KONAN].notna().sum(axis=1)

cohort["Slit_lamp"] = np.select(
    [n_slit_present == 0, n_slit_present == len(SLIT_LAMP)],
    ["Missing", "Complete"], default="Partial",
)
cohort["Konan"] = np.select(
    [n_konan_present == 0, n_konan_present == len(KONAN)],
    ["Missing", "Complete"], default="Partial",
)

slit_missing = (n_slit_present == 0).to_numpy()
konan_missing = (n_konan_present == 0).to_numpy()

for column in ["Slit_lamp", "Konan"]:
    table = pd.crosstab(cohort[column], cohort[ms.TARGET])[ms.CLASSES]
    table["Total"] = table.sum(axis=1)
    shares = pd.crosstab(cohort[column], cohort[ms.TARGET], normalize="index")[ms.CLASSES]
    chi2, p, _, _ = stats.chi2_contingency(table[ms.CLASSES])
    print(f"{column.replace('_', '-')} availability:")
    print(pd.concat([table, shares.add_suffix(" %").mul(100).round(1)], axis=1).to_string())
    print(f"Grade distribution differs by availability: chi2 = {chi2:.1f}, p = {p:.4f}\n")

print("Both missing:", int((slit_missing & konan_missing).sum()))

Slit-lamp availability:
Tissue_Grade_Clean   A+    A   B  Total  A+ %   A %   B %
Slit_lamp                                                
Complete             98   75  26    199  49.2  37.7  13.1
Missing             253  118  87    458  55.2  25.8  19.0
Partial             179  121  52    352  50.9  34.4  14.8
Grade distribution differs by availability: chi2 = 13.2, p = 0.0102

Konan availability:
Tissue_Grade_Clean   A+    A    B  Total  A+ %   A %   B %
Konan                                                     
Complete            497  277  150    924  53.8  30.0  16.2
Missing              33   37   15     85  38.8  43.5  17.6
Grade distribution differs by availability: chi2 = 8.0, p = 0.0183

Both missing: 4


### Is missingness tied to time period?

If slit-lamp findings are missing mainly in certain years, the natural comparison (C1) also compares time periods, and any difference may reflect changes in grading practice rather than the value of images.

In [4]:
year = pd.to_datetime(cohort["Konan_Exam_Date"], errors="coerce").dt.year
year = year.fillna(cohort["Tissue_ID"].str.extract(r"NEB-\s*(\d{4})")[0].astype(float))
cohort["Year"] = year.astype("Int64")

print("Share of tissues missing each source, by year:")
print(pd.DataFrame({
    "Tissues": cohort.groupby("Year").size(),
    "Slit-lamp missing %": cohort.groupby("Year")["Slit_lamp"].apply(lambda s: (s == "Missing").mean() * 100),
    "Konan missing %": cohort.groupby("Year")["Konan"].apply(lambda s: (s == "Missing").mean() * 100),
}).round(1).to_string())

pair_status = cohort.groupby("Donor_Group")["Slit_lamp"].agg(lambda s: s.nunique() if len(s) > 1 else np.nan).dropna()
print(f"\nDonor pairs whose eyes differ in slit-lamp availability: {(pair_status > 1).sum()} of {len(pair_status)}")

Share of tissues missing each source, by year:
      Tissues  Slit-lamp missing %  Konan missing %
Year                                               
2020      128                  0.0             39.1
2021       12                  0.0              8.3
2022       58                  0.0              6.9
2023      455                100.0              0.9
2024      171                  1.8              7.6
2025      185                  0.0              7.0

Donor pairs whose eyes differ in slit-lamp availability: 62 of 239


## 3. Model Arms

- **Primary arms** (as notebook 07): Tabular and Hybrid, trained on all 24 features with natural missingness. Their predictions on unmasked held-out tissues serve C1 and C2; on masked held-out tissues, C3 and C4.
- **Pre-examination arms** (C5): the same two arms, but with the slit-lamp features removed from training and testing.

In [5]:
def make_arm(tabular_features, with_image):
    numeric = [f for f in tabular_features if f in ms.NUMERIC_FEATURES]
    binary = [f for f in tabular_features if f in ms.CLINICAL_BINARY_FEATURES]
    categorical = [f for f in tabular_features if f in ms.CLINICAL_CATEGORICAL_FEATURES]
    branches = [("tabular", ms.make_preprocessor(numeric, binary, categorical), tabular_features)]
    if with_image:
        branches.append(("image", Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=PCA_COMPONENTS, random_state=SEED)),
        ]), EMBEDDING_COLUMNS))
    return Pipeline([
        ("features", ColumnTransformer(branches, remainder="drop")),
        ("classifier", RandomForestClassifier(**RF_PARAMS)),
    ])

ARMS = {
    "Tabular": make_arm(ms.ALL_FEATURES, with_image=False),
    "Hybrid": make_arm(ms.ALL_FEATURES, with_image=True),
    "Konan only": make_arm(KONAN, with_image=False),
    "Konan + image": make_arm(KONAN, with_image=True),
}


def mask(X_part, columns):
    masked = X_part.copy()
    masked[columns] = np.nan
    return masked

## 4. Cross-Validated Predictions

In [6]:
records = []
fold_sizes = []
start = time.time()

for repeat in range(ms.N_REPEATS):
    for fold, (train, test) in enumerate(ms.iter_donor_folds(folds, cohort["Tissue_ID"], repeat)):
        train_donors = set(cohort.loc[train, "Donor_Group"])
        assert train_donors.isdisjoint(cohort.loc[test, "Donor_Group"]), "Donor leak"
        fold_sizes.append((len(train), len(test)))

        fitted = {name: clone(arm).fit(X.iloc[train], y[train]) for name, arm in ARMS.items()}
        X_test = X.iloc[test]

        predictions = {
            ("Unmasked", "Tabular"): fitted["Tabular"].predict(X_test),
            ("Unmasked", "Hybrid"): fitted["Hybrid"].predict(X_test),
            ("Slit-lamp masked", "Tabular"): fitted["Tabular"].predict(mask(X_test, SLIT_LAMP)),
            ("Slit-lamp masked", "Hybrid"): fitted["Hybrid"].predict(mask(X_test, SLIT_LAMP)),
            ("Konan masked", "Tabular"): fitted["Tabular"].predict(mask(X_test, KONAN)),
            ("Konan masked", "Hybrid"): fitted["Hybrid"].predict(mask(X_test, KONAN)),
            ("Pre-examination", "Tabular"): fitted["Konan only"].predict(X_test),
            ("Pre-examination", "Hybrid"): fitted["Konan + image"].predict(X_test),
        }
        for (condition, arm), pred in predictions.items():
            records.append(pd.DataFrame({
                "Repeat": repeat, "Fold": fold, "Row": test,
                "Condition": condition, "Arm": arm, "Pred": pred,
            }))

predictions = pd.concat(records, ignore_index=True)
N_TRAIN, N_TEST = np.mean(fold_sizes, axis=0)
print(f"{len(fold_sizes)} folds, {len(ARMS)} arms each, in {(time.time() - start) / 60:.1f} min")

25 folds, 4 arms each, in 1.7 min


### Reproducibility check

The unmasked predictions should match the out-of-fold predictions saved by notebook 07.

In [7]:
saved = pd.read_csv(NB07_OOF_PATH)
for arm, saved_column in [("Tabular", "Tabular"), ("Hybrid", f"Hybrid (PCA {PCA_COMPONENTS})")]:
    ours = (
        predictions.query("Condition == 'Unmasked' and Arm == @arm")
        .assign(Tissue_ID=lambda d: cohort["Tissue_ID"].to_numpy()[d["Row"]])
        .merge(saved[["Tissue_ID", "Repeat", saved_column]], on=["Tissue_ID", "Repeat"])
    )
    agreement = (ours["Pred"] == ours[saved_column]).mean()
    print(f"{arm}: {agreement:.1%} of {len(ours)} predictions match notebook 07")

Tabular: 100.0% of 5045 predictions match notebook 07
Hybrid: 100.0% of 5045 predictions match notebook 07


## 5. Paired Comparison Helpers

In [8]:
def b_recall(true, pred):
    if not (true == "B").any():
        return np.nan
    return recall_score(true, pred, labels=["B"], average="macro", zero_division=0)

METRICS = {
    "Macro F1": lambda t, p: f1_score(t, p, average="macro", zero_division=0),
    "Balanced accuracy": balanced_accuracy_score,
    "Recall B": b_recall,
}


def corrected_ttest(differences):
    d = np.asarray(differences, dtype=float)
    d = d[~np.isnan(d)]
    n = len(d)
    se = np.sqrt((1 / n + N_TEST / N_TRAIN) * d.var(ddof=1))
    t = d.mean() / se if se > 0 else 0.0
    p = 2 * stats.t.sf(abs(t), df=n - 1)
    half = stats.t.ppf(0.975, df=n - 1) * se
    return d.mean(), d.mean() - half, d.mean() + half, p, n


def fold_metrics(condition, rows_mask):
    # Fold-level metrics for both arms on the same tissues.
    rows = np.flatnonzero(rows_mask)
    subset = predictions[(predictions["Condition"] == condition) & predictions["Row"].isin(rows)]
    out = []
    for (repeat, fold, arm), group in subset.groupby(["Repeat", "Fold", "Arm"]):
        true = y[group["Row"].to_numpy()]
        out.append({
            "Repeat": repeat, "Fold": fold, "Arm": arm, "N": len(group),
            **{m: fn(true, group["Pred"].to_numpy()) for m, fn in METRICS.items()},
        })
    return pd.DataFrame(out)


def compare(condition, rows_mask, label):
    scores = fold_metrics(condition, rows_mask)
    wide = scores.pivot_table(index=["Repeat", "Fold"], columns="Arm", values=list(METRICS))
    rows = []
    for metric in METRICS:
        mean, low, high, p, n = corrected_ttest(wide[(metric, "Hybrid")] - wide[(metric, "Tabular")])
        rows.append({
            "Comparison": label, "Metric": metric,
            "Tabular": wide[(metric, "Tabular")].mean(),
            "Hybrid": wide[(metric, "Hybrid")].mean(),
            "Difference": mean, "CI low": low, "CI high": high, "p": p,
            "Folds": n,
        })
    tissues = int(rows_mask.sum())
    min_fold = int(scores.groupby(["Repeat", "Fold"])["N"].first().min())
    return pd.DataFrame(rows), tissues, min_fold


def show(table, tissues, min_fold):
    print(f"{table['Comparison'].iloc[0]} — {tissues} tissues, smallest fold subgroup {min_fold}")
    print(table.drop(columns="Comparison").round(4).to_string(index=False), "\n")


everyone = np.ones(len(cohort), dtype=bool)
results = {}

## 6. Natural Missingness (C1, C2)

Both arms are the primary models evaluated on unmasked held-out tissues; only the evaluation subgroup changes. The complementary subgroup is shown for reference, together with the interaction (whether the hybrid's gain differs between subgroups).

In [9]:
def interaction(condition, group_a, group_b):
    # Difference in hybrid gain (macro F1) between two subgroups, per fold.
    gains = []
    for rows_mask in [group_a, group_b]:
        wide = fold_metrics(condition, rows_mask).pivot_table(
            index=["Repeat", "Fold"], columns="Arm", values="Macro F1"
        )
        gains.append(wide["Hybrid"] - wide["Tabular"])
    mean, low, high, p, _ = corrected_ttest(gains[0] - gains[1])
    return f"gain difference {mean:+.4f} [{low:+.4f}, {high:+.4f}], p = {p:.3f}"


for label, rows_mask, condition in [
    ("C1: slit-lamp missing", slit_missing, "Unmasked"),
    ("Ref: slit-lamp available", ~slit_missing, "Unmasked"),
]:
    table, n, m = compare(condition, rows_mask, label)
    results[label] = table
    show(table, n, m)
print("Interaction (missing vs available):", interaction("Unmasked", slit_missing, ~slit_missing))

C1: slit-lamp missing — 458 tissues, smallest fold subgroup 78
           Metric  Tabular  Hybrid  Difference  CI low  CI high      p  Folds
         Macro F1   0.9921  0.9702     -0.0219 -0.0445   0.0006 0.0564     25
Balanced accuracy   0.9915  0.9635     -0.0280 -0.0557  -0.0002 0.0483     25
         Recall B   0.9850  0.9439     -0.0411 -0.0946   0.0125 0.1265     25 



Ref: slit-lamp available — 551 tissues, smallest fold subgroup 101
           Metric  Tabular  Hybrid  Difference  CI low  CI high      p  Folds
         Macro F1   0.7282  0.7641      0.0359 -0.0044   0.0762 0.0785     25
Balanced accuracy   0.7150  0.7371      0.0221 -0.0176   0.0617 0.2617     25
         Recall B   0.6979  0.6780     -0.0200 -0.0720   0.0320 0.4358     25 



Interaction (missing vs available): gain difference -0.0578 [-0.1094, -0.0063], p = 0.030


In [10]:
for label, rows_mask, condition in [
    ("C2: Konan missing", konan_missing, "Unmasked"),
    ("Ref: Konan available", ~konan_missing, "Unmasked"),
]:
    table, n, m = compare(condition, rows_mask, label)
    results[label] = table
    show(table, n, m)
print("Interaction (missing vs available):", interaction("Unmasked", konan_missing, ~konan_missing))

C2: Konan missing — 85 tissues, smallest fold subgroup 11
           Metric  Tabular  Hybrid  Difference  CI low  CI high      p  Folds
         Macro F1   0.4702  0.4025     -0.0677 -0.2350   0.0997 0.4121     25
Balanced accuracy   0.4917  0.4514     -0.0403 -0.1956   0.1151 0.5977     25
         Recall B   0.3694  0.0535     -0.3160 -0.7330   0.1010 0.1306     24 



Ref: Konan available — 924 tissues, smallest fold subgroup 178
           Metric  Tabular  Hybrid  Difference  CI low  CI high      p  Folds
         Macro F1   0.8793  0.8939      0.0146 -0.0089   0.0381 0.2111     25
Balanced accuracy   0.8727  0.8818      0.0090 -0.0123   0.0303 0.3911     25
         Recall B   0.8997  0.8961     -0.0036 -0.0250   0.0178 0.7322     25 



Interaction (missing vs available): gain difference -0.0823 [-0.2583, +0.0937], p = 0.344


## 7. Controlled Masking (C3, C4, C5)

C3 and C4 use the primary models and remove one source from held-out tissues that originally had it, so the same tissues are compared with and without that information. C5 trains and tests without slit-lamp findings, matching the pre-examination scenario.

In [11]:
has_slit = ~slit_missing
has_konan = ~konan_missing

for label, rows_mask, condition in [
    ("C3: slit-lamp masked at test", has_slit, "Slit-lamp masked"),
    ("C4: Konan masked at test", has_konan, "Konan masked"),
    ("C5: pre-examination (no slit-lamp)", everyone, "Pre-examination"),
]:
    table, n, m = compare(condition, rows_mask, label)
    results[label] = table
    show(table, n, m)

C3: slit-lamp masked at test — 551 tissues, smallest fold subgroup 101
           Metric  Tabular  Hybrid  Difference  CI low  CI high      p  Folds
         Macro F1   0.6782  0.7398      0.0616  0.0239   0.0993 0.0025     25
Balanced accuracy   0.7209  0.7105     -0.0104 -0.0493   0.0284 0.5839     25
         Recall B   0.8198  0.6804     -0.1395 -0.2386  -0.0403 0.0078     25 



C4: Konan masked at test — 924 tissues, smallest fold subgroup 178
           Metric  Tabular  Hybrid  Difference  CI low  CI high      p  Folds
         Macro F1   0.3057  0.3668      0.0610  0.0113   0.1108 0.0183     25
Balanced accuracy   0.3920  0.3901     -0.0019 -0.0448   0.0410 0.9279     25
         Recall B   0.6715  0.0419     -0.6296 -0.7159  -0.5433 0.0000     25 



C5: pre-examination (no slit-lamp) — 1009 tissues, smallest fold subgroup 199
           Metric  Tabular  Hybrid  Difference  CI low  CI high      p  Folds
         Macro F1    0.845  0.8502      0.0053 -0.0232   0.0337 0.7062     25
Balanced accuracy    0.833  0.8333      0.0003 -0.0238   0.0245 0.9776     25
         Recall B    0.818  0.8326      0.0146 -0.0072   0.0364 0.1800     25 



### Cost of losing each source

Macro F1 on the same tissues before and after masking, for each arm. A hybrid that is more robust to missing information would lose less.

In [12]:
robustness = []
for source, rows_mask, condition in [
    ("Slit-lamp findings", has_slit, "Slit-lamp masked"),
    ("Konan measurements", has_konan, "Konan masked"),
]:
    before = fold_metrics("Unmasked", rows_mask).pivot_table(index=["Repeat", "Fold"], columns="Arm", values="Macro F1")
    after = fold_metrics(condition, rows_mask).pivot_table(index=["Repeat", "Fold"], columns="Arm", values="Macro F1")
    drop = before - after
    mean, low, high, p, _ = corrected_ttest(drop["Tabular"] - drop["Hybrid"])
    robustness.append({
        "Masked source": source,
        "Tabular before": before["Tabular"].mean(), "Tabular after": after["Tabular"].mean(),
        "Hybrid before": before["Hybrid"].mean(), "Hybrid after": after["Hybrid"].mean(),
        "Drop saved by images": mean,
        "95% CI": f"[{low:+.4f}, {high:+.4f}]", "p": p,
    })
print(pd.DataFrame(robustness).round(4).to_string(index=False))

     Masked source  Tabular before  Tabular after  Hybrid before  Hybrid after  Drop saved by images             95% CI      p
Slit-lamp findings          0.7282         0.6782         0.7641        0.7398                0.0257 [-0.0227, +0.0741] 0.2845
Konan measurements          0.8793         0.3057         0.8939        0.3668                0.0464 [-0.0006, +0.0935] 0.0529


## 8. Per-Grade Recall

Pooled over all five repeats. Grade B recall matters most clinically: a missed B means poorer tissue rated as better.

In [13]:
def pooled_recall(condition, rows_mask):
    rows = np.flatnonzero(rows_mask)
    subset = predictions[(predictions["Condition"] == condition) & predictions["Row"].isin(rows)]
    out = {}
    for arm, group in subset.groupby("Arm"):
        true = y[group["Row"].to_numpy()]
        out[arm] = recall_score(true, group["Pred"], labels=ms.CLASSES, average=None, zero_division=0)
    return pd.DataFrame(out, index=[f"Recall {c}" for c in ms.CLASSES])[["Tabular", "Hybrid"]]

recall_tables = {
    "C1 slit-lamp missing": pooled_recall("Unmasked", slit_missing),
    "C2 Konan missing": pooled_recall("Unmasked", konan_missing),
    "C3 slit-lamp masked": pooled_recall("Slit-lamp masked", has_slit),
    "C4 Konan masked": pooled_recall("Konan masked", has_konan),
    "C5 pre-examination": pooled_recall("Pre-examination", everyone),
}
print(pd.concat(recall_tables, axis=1).round(3).to_string())

          C1 slit-lamp missing        C2 Konan missing        C3 slit-lamp masked        C4 Konan masked        C5 pre-examination       
                       Tabular Hybrid          Tabular Hybrid             Tabular Hybrid         Tabular Hybrid            Tabular Hybrid
Recall A+                0.998  0.994            0.436  0.661               0.841  0.959           0.193  0.833              0.883  0.926
Recall A                 0.992  0.951            0.670  0.568               0.503  0.493           0.313  0.298              0.798  0.741
Recall B                 0.984  0.943            0.360  0.053               0.821  0.685           0.672  0.043              0.818  0.833


## 9. Multiple-Comparison Correction

Holm correction across the five pre-specified macro F1 comparisons.

In [14]:
primary = pd.concat([
    results[k].query("Metric == 'Macro F1'")
    for k in results if k.startswith("C")
]).reset_index(drop=True)

order = np.argsort(primary["p"].to_numpy())
m = len(primary)
holm = np.empty(m)
running = 0.0
for rank, idx in enumerate(order):
    running = max(running, (m - rank) * primary.loc[idx, "p"])
    holm[idx] = min(running, 1.0)
primary["p (Holm)"] = holm
primary["95% CI"] = primary.apply(lambda r: f"[{r['CI low']:+.4f}, {r['CI high']:+.4f}]", axis=1)

print(primary[["Comparison", "Tabular", "Hybrid", "Difference", "95% CI", "p", "p (Holm)"]]
      .round(4).to_string(index=False))

all_results = pd.concat(results.values(), ignore_index=True)
all_results.to_csv(ms.PROCESSED_DIR / "missing_information_results.csv", index=False)
print("\nSaved:", ms.PROCESSED_DIR / "missing_information_results.csv")

                        Comparison  Tabular  Hybrid  Difference             95% CI      p  p (Holm)
             C1: slit-lamp missing   0.9921  0.9702     -0.0219 [-0.0445, +0.0006] 0.0564    0.1692
                 C2: Konan missing   0.4702  0.4025     -0.0677 [-0.2350, +0.0997] 0.4121    0.8241
      C3: slit-lamp masked at test   0.6782  0.7398      0.0616 [+0.0239, +0.0993] 0.0025    0.0127
          C4: Konan masked at test   0.3057  0.3668      0.0610 [+0.0113, +0.1108] 0.0183    0.0732
C5: pre-examination (no slit-lamp)   0.8450  0.8502      0.0053 [-0.0232, +0.0337] 0.7062    0.8241

Saved: C:\Users\pulit\OneDrive\Desktop\University\Y4S1\research\xai-cornea-grading\ml\grading\data\processed\missing_information_results.csv


## 10. Year Confound (added after inspecting Sections 2 and 6)

Section 2 showed that slit-lamp findings are missing for every 2023 tissue and almost no others, and Section 6 showed near-perfect tabular performance on that subgroup. This section was added after those results to describe them. It is diagnostic, not one of the pre-specified comparisons, and it describes patterns in the recorded data; it does not establish how the grades were assigned.

### 10.1 How closely do recorded grades follow ECD thresholds?

Reference thresholds: B if ECD < 2250, A if 2250 ≤ ECD < 2500, A+ if ECD ≥ 2500.

In [15]:
era_2023 = (cohort["Year"] == 2023).to_numpy()
ecd = cohort["ECD_CD_cells_per_mm2"]
ecd_rule = np.where(ecd >= 2500, "A+", np.where(ecd >= 2250, "A", "B"))

rule_check = pd.DataFrame({"Year": cohort["Year"], "Agrees": ecd_rule == y})[ecd.notna().to_numpy()]
print("Agreement between specialist grade and ECD rule, by year:")
print(rule_check.groupby("Year")["Agrees"].agg(["size", "mean"]).rename(
    columns={"size": "Tissues", "mean": "Agreement"}).round(3).to_string())

print("\nECD range by grade:")
print(cohort.assign(Era=np.where(era_2023, "2023", "Other years"))
      .groupby(["Era", ms.TARGET])["ECD_CD_cells_per_mm2"]
      .describe()[["count", "min", "50%", "max"]].round(0).to_string())

exceptions = cohort[era_2023 & ecd.notna().to_numpy() & (ecd_rule != y)]
print(f"\n2023 tissues not following the rule: {len(exceptions)}; "
      f"grades {exceptions[ms.TARGET].value_counts().to_dict()}, "
      f"eyes {exceptions['Eye'].value_counts().to_dict()}")

Agreement between specialist grade and ECD rule, by year:


      Tissues  Agreement
Year                    
2020       78      0.756
2021       11      0.818
2022       54      0.722
2023      451      0.951
2024      158      0.823
2025      172      0.837

ECD range by grade:
                                count     min     50%     max
Era         Tissue_Grade_Clean                               
2023        A                   117.0  2252.0  2364.0  3759.0
            A+                  251.0  2500.0  2725.0  4032.0
            B                    83.0   951.0  2020.0  2247.0
Other years A                   160.0  2252.0  2475.0  4484.0
            A+                  246.0  2326.0  2805.0  3676.0
            B                    67.0   788.0  2033.0  3448.0

2023 tissues not following the rule: 22; grades {'A': 22}, eyes {'Right': 22}


### 10.2 Performance by era

Unmasked primary models, evaluated within each era.

In [16]:
for label, rows_mask in [("2023", era_2023), ("Other years", ~era_2023)]:
    table, n, m = compare("Unmasked", rows_mask, f"Era: {label}")
    show(table, n, m)

overlap = pd.crosstab(np.where(era_2023, "2023", "Other years"), cohort["Slit_lamp"])
print("Era vs slit-lamp availability:")
print(overlap)

Era: 2023 — 455 tissues, smallest fold subgroup 78
           Metric  Tabular  Hybrid  Difference  CI low  CI high      p  Folds
         Macro F1   0.9920  0.9705     -0.0215 -0.0437   0.0006 0.0556     25
Balanced accuracy   0.9914  0.9639     -0.0275 -0.0544  -0.0005 0.0462     25
         Recall B   0.9850  0.9439     -0.0411 -0.0946   0.0125 0.1265     25 



Era: Other years — 554 tissues, smallest fold subgroup 102
           Metric  Tabular  Hybrid  Difference  CI low  CI high      p  Folds
         Macro F1   0.7294  0.7649      0.0355 -0.0042   0.0751 0.0776     25
Balanced accuracy   0.7161  0.7377      0.0216 -0.0175   0.0608 0.2650     25
         Recall B   0.6979  0.6780     -0.0200 -0.0720   0.0320 0.4358     25 

Era vs slit-lamp availability:
Slit_lamp    Complete  Missing  Partial
row_0                                  
2023                0      455        0
Other years       199        3      352


### 10.3 Does masking push the tabular model toward the 2023 rule?

If the model learned "no slit-lamp findings means 2023-style grading", masking findings on tissues from other years should make its predictions agree more with the ECD rule.

In [17]:
other_with_ecd = np.flatnonzero(~era_2023 & has_slit & ecd.notna().to_numpy())
rows = []
for condition in ["Unmasked", "Slit-lamp masked"]:
    for arm in ["Tabular", "Hybrid"]:
        subset = predictions[
            (predictions["Condition"] == condition) & (predictions["Arm"] == arm)
            & predictions["Row"].isin(other_with_ecd)
        ]
        rows.append({
            "Condition": condition, "Arm": arm,
            "Agreement with ECD rule": (subset["Pred"].to_numpy() == ecd_rule[subset["Row"]]).mean(),
            "Agreement with specialist": (subset["Pred"].to_numpy() == y[subset["Row"]]).mean(),
        })
print(f"Tissues from other years with findings and ECD: {len(other_with_ecd)}")
print(pd.DataFrame(rows).round(3).to_string(index=False))

Tissues from other years with findings and ECD: 470
       Condition     Arm  Agreement with ECD rule  Agreement with specialist
        Unmasked Tabular                    0.850                      0.769
        Unmasked  Hybrid                    0.870                      0.808
Slit-lamp masked Tabular                    0.946                      0.818
Slit-lamp masked  Hybrid                    0.947                      0.813


### 10.4 Can the images identify the era?

If the image embeddings recognise 2023 tissues, the hybrid could avoid the 2023 rule under masking by detecting the era rather than by reading clinical information from the image.

In [18]:
from sklearn.metrics import roc_auc_score

era_auc = []
for repeat in range(ms.N_REPEATS):
    for train, test in ms.iter_donor_folds(folds, cohort["Tissue_ID"], repeat):
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=PCA_COMPONENTS, random_state=SEED)),
            ("classifier", RandomForestClassifier(**RF_PARAMS)),
        ]).fit(embeddings[train], era_2023[train])
        era_auc.append(roc_auc_score(era_2023[test], model.predict_proba(embeddings[test])[:, 1]))

print(f"Image embeddings predicting 2023 vs other years: AUC {np.mean(era_auc):.3f} ± {np.std(era_auc):.3f}")

Image embeddings predicting 2023 vs other years: AUC 0.621 ± 0.042


## Summary

### Main finding: slit-lamp missingness coincides with a distinct pattern in 2023

- Slit-lamp findings are missing for **all 455 tissues from 2023** and almost no others (Section 2). The missing findings cannot be recovered.
- In 2023, recorded grades agree with fixed ECD thresholds (B < 2250, A 2250–2499, A+ ≥ 2500) for **95%** of tissues, compared with 72–84% in other years (Section 10.1).
- The tabular model reaches macro F1 **0.992 on 2023 tissues** and **0.729 on other years**. The pooled estimates in notebooks 05–07 (~0.85) combine these two subsets.
- These are patterns in the recorded data. They do not show why the patterns arose: they are consistent with, for example, a change in protocol or documentation, but they do not prove that 2023 grades were assigned from ECD alone.
- Notebook 06 found that missingness indicators do not predict the grade (correct), but missingness does identify the 2023 subset.

### Pre-specified comparisons (Hybrid − Tabular, macro F1)

| ID | Comparison | Difference [95% CI] | p (Holm) | Interpretation |
|---|---|---|---|---|
| C1 | Natural: slit-lamp missing | −0.022 [−0.045, +0.001] | 0.17 | Not interpretable as image value: same tissues as the 2023 subset, tabular at ceiling (0.99) |
| C2 | Natural: Konan missing (85 tissues) | −0.068 [−0.235, +0.100] | 0.82 | Too few tissues; no evidence |
| C3 | Slit-lamp masked at test | **+0.062 [+0.024, +0.099]** | **0.013** | Significant, but confounded (below) |
| C4 | Konan masked at test | +0.061 [+0.011, +0.111] | 0.073 | Both arms fail (0.31 vs 0.37); hybrid B recall 0.04 |
| C5 | Pre-examination (no slit-lamp in training) | +0.005 [−0.023, +0.034] | 0.82 | No benefit |

**C3 does not show that images replace slit-lamp findings:**

- Masking shifts the tabular model toward the 2023 pattern: its agreement with the ECD thresholds rises from 0.85 to 0.95 on tissues from other years (Section 10.3). The model treats a tissue without findings like a 2023 tissue, so the hybrid's advantage under masking reflects how the two arms respond to this artificial shift.
- Balanced accuracy is unchanged (−0.010) and **grade B recall falls from 0.82 to 0.68** (p = 0.008). The macro F1 gain comes from more A+ predictions, at the cost of missing poorer tissue.
- The image embeddings only weakly identify 2023 tissues (AUC 0.62), so recognising the year is at most a partial explanation of the hybrid's behaviour.

**C5 is the cleanest test of the hypothesis.** It trains and tests without slit-lamp findings, so there is no masking artefact, and it shows no gain from frozen ImageNet embeddings.

**C4:** images cannot stand in for Konan measurements. Both arms collapse, and the hybrid almost never predicts B.

### Exploratory signal (not pre-specified)

On tissues from years other than 2023, where recorded grades depart more often from the ECD thresholds, the hybrid gains **+0.036 macro F1 [−0.004, +0.075], p = 0.078** (unadjusted), and the gain is larger than in 2023 (interaction p = 0.03). This was found after seeing the data, so it is hypothesis-generating only.

### Conclusions

1. Frozen ResNet-50 embeddings did not reliably improve grading when slit-lamp findings or Konan measurements were unavailable. The one significant result (C3) is explained by a masking artefact and comes with a clinically unfavourable drop in grade B recall.
2. The recorded data contain two distinct patterns. Pooled performance overstates performance on the subset where grades depart from ECD thresholds. Results should be reported overall and by year.

### Next steps

- Treat the 2023 slit-lamp data as permanently missing and document the pattern as a limitation and a finding.
- Run a separately pre-specified tabular vs hybrid comparison on non-2023 tissues, with settings fixed before results are examined.
- Do not use test-time masking of slit-lamp findings as evidence of robustness while missingness is tied to the 2023 subset.